# 프로젝트 2. Appliance Vision Tuning

`day2`의 vision / ONNX 흐름을 바탕으로 제품형 상태 판독 모델을 구현하는 실습입니다.

## 목표
- 간단한 이미지 분류 모델 학습
- `predict(image) -> {label, confidence, recommendation}` 형태로 래핑
- 검증용 10장 inference 결과 테이블 생성
- ONNX export까지 연결

## 현재 제공 데이터
- synthetic `filter_clean` / `filter_dirty` 샘플 이미지
- `manifest.csv`로 train / val split 제공


## 진행 순서

1. 데이터 확인
2. Dataset / DataLoader 준비
3. `build_model()` 구현
4. `train_one_epoch()`와 `evaluate()` 구현
5. `predict()` 구현
6. 10장 inference 테이블 생성
7. `export_to_onnx()` 구현

모델은 `ResNet18`을 권장합니다.


In [ ]:
# 선택: 최초 1회만 실행
# %pip install -r ../requirements.txt


In [ ]:
from __future__ import annotations

import csv
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
ARTIFACT_DIR = BASE_DIR / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

manifest = pd.read_csv(DATA_DIR / "manifest.csv")
manifest.head()


In [ ]:
def set_seed(seed: int = 7):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


## Step 1. Dataset / Transform 준비

이 부분은 그대로 사용해도 됩니다.


In [ ]:
LABEL_TO_ID = {"clean": 0, "dirty": 1}
ID_TO_LABEL = {0: "clean", 1: "dirty"}
RECOMMENDATIONS = {
    "clean": "필터 상태가 양호합니다. 정기 점검만 유지하세요.",
    "dirty": "필터 오염 가능성이 높습니다. 청소 또는 교체를 권장합니다.",
}


def build_transforms(train: bool, image_size: int = 64):
    ops = [transforms.Resize((image_size, image_size))]
    if train:
        ops.append(transforms.RandomHorizontalFlip())
    ops.extend(
        [
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ]
    )
    return transforms.Compose(ops)


class ApplianceVisionDataset(Dataset):
    def __init__(self, manifest_df: pd.DataFrame, split: str, image_size: int = 64):
        self.rows = manifest_df[manifest_df["split"] == split].reset_index(drop=True)
        self.transform = build_transforms(train=split == "train", image_size=image_size)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index: int):
        row = self.rows.iloc[index]
        image = Image.open(DATA_DIR / row["path"]).convert("RGB")
        image = self.transform(image)
        label = LABEL_TO_ID[row["label"]]
        return image, label, row["path"]


## Step 2. 모델 정의

요구사항:
- `ResNet18` 사용
- 마지막 FC를 2-class 분류기로 교체
- 반환값은 `torch.nn.Module`


In [ ]:
def build_model(num_classes: int = 2) -> nn.Module:
    # TODO:
    # 1) models.resnet18(...) 생성
    # 2) model.fc를 num_classes에 맞게 교체
    raise NotImplementedError("build_model()를 구현하세요.")


## Step 3. 학습 / 평가 루프 구현

요구사항:
- `train_one_epoch`: forward, loss, backward, optimizer step
- `evaluate`: accuracy 계산


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    # TODO:
    # - model.train()
    # - batch loop
    # - loss 평균 반환
    raise NotImplementedError("train_one_epoch()를 구현하세요.")


def evaluate(model, loader, device):
    # TODO:
    # - model.eval()
    # - accuracy 계산
    # - 필요하면 prediction 결과도 같이 반환
    raise NotImplementedError("evaluate()를 구현하세요.")


## Step 4. 학습 실행

TODO 함수들을 구현한 뒤 아래 셀을 실행하세요.


In [ ]:
train_dataset = ApplianceVisionDataset(manifest, split="train", image_size=64)
val_dataset = ApplianceVisionDataset(manifest, split="val", image_size=64)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0)

print("train:", len(train_dataset), "val:", len(val_dataset))


In [ ]:
# TODO 구현 후 실행하세요.
#
# model = build_model(num_classes=2).to(device)
# criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
#
# for epoch in range(3):
#     train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
#     val_acc = evaluate(model, val_loader, device)
#     print(f"epoch={epoch+1} train_loss={train_loss:.4f} val_acc={val_acc:.4f}")


## Step 5. 제품형 JSON 출력 함수 구현

요구사항:
- 입력: 이미지 경로
- 출력: `{label, confidence, recommendation}`


In [ ]:
def predict(image_path: str | Path, model, image_size: int = 64) -> dict[str, object]:
    # TODO:
    # 1) 이미지 로드 및 transform
    # 2) softmax confidence 계산
    # 3) recommendation 매핑
    raise NotImplementedError("predict()를 구현하세요.")


## Step 6. 10장 inference 결과 테이블 생성


In [ ]:
# TODO 구현 후 실행하세요.
#
# demo_rows = []
# val_paths = manifest[manifest["split"] == "val"]["path"].tolist()[:10]
# for rel_path in val_paths:
#     result = predict(DATA_DIR / rel_path, model, image_size=64)
#     demo_rows.append({"image_path": rel_path, **result})
#
# demo_df = pd.DataFrame(demo_rows)
# demo_df
# demo_df.to_csv(ARTIFACT_DIR / "demo_predictions.csv", index=False)


## Step 7. ONNX Export 구현

요구사항:
- `torch.onnx.export`
- dummy input shape: `(1, 3, 64, 64)`
- output file: `artifacts/filter_classifier.onnx`


In [ ]:
import onnx


def export_to_onnx(model, output_path: str | Path, image_size: int = 64):
    # TODO:
    # 1) model.eval()
    # 2) dummy_input 생성
    # 3) torch.onnx.export 호출
    # 4) onnx.checker.check_model 검증
    raise NotImplementedError("export_to_onnx()를 구현하세요.")


## Bonus

아래 중 하나를 추가하면 기존 학습의 흐름과 더 잘 이어집니다.
- confusion matrix 시각화
- class imbalance 대응
- `onnxruntime` 추론 코드 추가
- `predict()` 결과를 Agent 프로젝트의 recommendation 입력으로 연결
